# Prediction Market Expected Value Framework

This notebook walks through the full toolchain for evaluating prediction market contracts:

| Step | Tool | Question it answers |
|------|------|---------------------|
| 1 | `implied_probability` | What does the market believe? |
| 2 | `bayesian_update` | How should new information shift my estimate? |
| 3 | `expected_value` | How much do I expect to profit per contract? |
| 4 | `edge` | How far am I from the market consensus? |
| 5 | `kelly_fraction` | How much of my bankroll should I risk? |

---

## 1. Implied Probability

Prediction market contracts pay **\$100 if the event occurs**, **\$0 if not**.
The price you pay to enter is a direct reflection of what the market thinks the probability is:

$$P_{implied} = \frac{\text{market price}}{\text{payout}}$$

**Example:** Contract trading at **\$40** → market implies **40% probability** of the event.

In [ ]:
def implied_probability(market_price, payout=100):
    """
    Extract the market-implied probability from a contract price.

    Args:
        market_price: current price of the contract
        payout: what the contract pays if event occurs (default $100)

    Returns:
        float: implied probability (0 to 1)
    """
    return market_price / payout


# --- Demo ---
prices = [20, 35, 40, 55, 72]
for p in prices:
    print(f"Contract @ ${p:>3} → implied prob: {implied_probability(p)*100:.0f}%")

## 2. Bayesian Update

When you receive an analyst signal, you shouldn't blindly replace your prior — you should **update it** using Bayes' theorem.

$$P(E | S) = \frac{P(S | E) \cdot P(E)}{P(S)}$$

Where:
- $P(E)$ = prior probability (your estimate before the signal)
- $P(S | E)$ = signal accuracy (how often the signal is right when the event occurs)
- $P(S | \neg E) = 1 - \text{signal\_accuracy}$ (how often signal fires when event does NOT occur)
- $P(S) = P(S|E) \cdot P(E) + P(S|\neg E) \cdot P(\neg E)$ = total probability of signal

**Intuition:** A 70%-accurate signal on a 35% prior shifts you to ~56% — meaningful, but not certain.

In [ ]:
def bayesian_update(prior, signal_accuracy):
    """
    Update probability using Bayes' theorem after receiving a bullish signal.

    Args:
        prior: prior probability of event occurring (0 to 1)
        signal_accuracy: P(signal correct) — how reliable the signal is

    Returns:
        float: posterior probability
    """
    p_signal_given_event    = signal_accuracy
    p_signal_given_no_event = 1 - signal_accuracy

    # Total probability of observing the signal
    p_signal = p_signal_given_event * prior + p_signal_given_no_event * (1 - prior)

    posterior = (p_signal_given_event * prior) / p_signal
    return round(posterior, 4)


# --- Demo: show how different signal accuracies affect the same prior ---
prior = 0.35
print(f"Prior probability: {prior*100:.0f}%\n")
print(f"{'Signal Accuracy':<20} {'Posterior':<15} {'Shift'}")
print("-" * 45)
for acc in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.90]:
    post = bayesian_update(prior, acc)
    shift = post - prior
    print(f"{acc*100:.0f}%{'':<17} {post*100:.1f}%{'':<10} +{shift*100:.1f}%")

## 3. Expected Value

Given your **true** probability estimate, expected value is what you expect to profit per contract:

$$EV = P_{true} \times \text{payout} - \text{market price}$$

A **positive EV** means you expect to profit over many repetitions.  
A **negative EV** means the market is pricing in an edge against you.

In [ ]:
def expected_value(true_prob, market_price, payout=100):
    """
    Calculate expected profit per contract.

    Args:
        true_prob: your estimated probability of event occurring
        market_price: what you pay per contract
        payout: contract payout if event occurs (default $100)

    Returns:
        float: expected profit per contract
    """
    ev = true_prob * payout
    return round(ev - market_price, 4)


# --- Demo: scan across different true prob estimates ---
market_price = 40
print(f"Market price: ${market_price} | Scanning your true probability estimate:\n")
print(f"{'True Prob':<15} {'EV / contract':<20} {'Signal'}")
print("-" * 45)
for prob in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    ev = expected_value(prob, market_price)
    signal = '✓ BUY' if ev > 0 else '✗ PASS' if ev == 0 else '✗ SELL'
    print(f"{prob*100:.0f}%{'':<12} ${ev:<19} {signal}")

## 4. Edge

**Edge** is simply the gap between your probability estimate and the market's:

$$\text{edge} = P_{true} - P_{implied}$$

- **Positive edge** → you think the event is more likely than the market does → buy
- **Negative edge** → market is overpricing the contract → sell or pass
- **Zero edge** → you agree with the market → no trade

In [ ]:
def edge(true_prob, market_price, payout=100):
    """Your probability edge over the market."""
    market_implied = implied_probability(market_price, payout)
    return round(true_prob - market_implied, 4)


# --- Demo ---
scenarios = [
    (0.55, 40, "Bullish — market underpricing"),
    (0.40, 40, "Neutral — agree with market"),
    (0.30, 40, "Bearish — market overpricing"),
]
print(f"{'True Prob':<12} {'Market Price':<15} {'Edge':<12} {'Description'}")
print("-" * 60)
for tp, mp, desc in scenarios:
    e = edge(tp, mp)
    print(f"{tp*100:.0f}%{'':<9} ${mp:<14} {e*100:+.1f}%{'':<7} {desc}")

## 5. Kelly Criterion

Kelly tells you the **optimal fraction of your bankroll** to allocate to maximize long-run growth.

$$f^* = \frac{bp - q}{b}$$

Where:
- $b$ = net odds = $\frac{\text{payout} - \text{price}}{\text{price}}$
- $p$ = your true probability of winning
- $q = 1 - p$ = probability of losing

**Practical note:** Many traders use **half-Kelly** (f/2) to reduce variance while retaining most of the edge.

In [ ]:
def kelly_fraction(true_prob, market_price, payout=100):
    """
    Kelly criterion: optimal fraction of bankroll to bet.

    Returns:
        float: fraction of bankroll to allocate (negative = no bet)
    """
    b = (payout - market_price) / market_price  # net odds
    p = true_prob
    q = 1 - p
    return round((b * p - q) / b, 4)


# --- Demo: Kelly vs Half-Kelly across probability estimates ---
market_price = 40
print(f"Market price: ${market_price}  |  Net odds: {(100-market_price)/market_price:.2f}x\n")
print(f"{'True Prob':<12} {'Full Kelly':<15} {'Half Kelly':<15} {'Action'}")
print("-" * 55)
for prob in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    k = kelly_fraction(prob, market_price)
    action = f'Bet {k*100:.1f}% of bankroll' if k > 0 else 'No bet'
    hk = max(0, k / 2)
    print(f"{prob*100:.0f}%{'':<9} {k*100:.1f}%{'':<12} {hk*100:.1f}%{'':<12} {action}")

## 6. Full Position Summary

The `position_summary` function wraps all the tools above into a single analysis block.
Optionally pass `signal_accuracy` to apply a Bayesian update before computing EV and sizing.

In [ ]:
def position_summary(true_prob, market_price, n_contracts, payout=100, signal_accuracy=None):
    """Full position analysis: Bayes update → EV → edge → Kelly → recommendation."""
    if signal_accuracy:
        updated_prob = bayesian_update(true_prob, signal_accuracy)
        print(f"Prior probability:     {true_prob*100:.1f}%")
        print(f"After signal update:   {updated_prob*100:.1f}%")
        true_prob = updated_prob

    mkt_implied = implied_probability(market_price, payout)
    ev          = expected_value(true_prob, market_price, payout)
    your_edge   = edge(true_prob, market_price, payout)
    kelly       = kelly_fraction(true_prob, market_price, payout)

    print(f"\n--- Position Summary ---")
    print(f"Market price:          ${market_price}")
    print(f"Market implied prob:   {mkt_implied*100:.1f}%")
    print(f"Your true prob:        {true_prob*100:.1f}%")
    print(f"Your edge:             {your_edge*100:.1f}%")
    print(f"EV per contract:       ${ev}")
    print(f"Kelly fraction:        {kelly*100:.1f}% of bankroll")
    print(f"Total EV ({n_contracts} contracts): ${round(ev * n_contracts, 2)}")
    print(f"Recommendation:        {'BUY' if your_edge > 0 else 'SELL' if your_edge < 0 else 'PASS'}")


print("=" * 40)
print("Scenario: Contract @ $40, prior 55%, signal accuracy 70%")
print("=" * 40)
position_summary(
    true_prob=0.55,
    market_price=40,
    n_contracts=50,
    payout=100,
    signal_accuracy=0.70
)

---
## Key Takeaways

1. **Price = probability** in prediction markets. A \$40 contract implies 40% odds.
2. **Bayes matters** — don't ignore your prior. A 70%-accurate signal on a 35% prior gives you ~56%, not 70%.
3. **Positive EV ≠ guaranteed profit** on any single trade — it's a long-run edge.
4. **Kelly maximizes long-run growth** but can be volatile. Half-Kelly is common in practice.
5. **Edge is the filter** — no edge, no trade.